# Are NBFNet's Path Explanations Actually Faithful? (Colab)

CAP 6938 term project, research track. Shadi Khodagholi, UCF.

This notebook runs the project end to end on Colab:

1. sets up the code, ULTRA's frozen `ultra_4g` checkpoint and the graphs,
2. runs the **live demo** twice, on two different draws of queries,
3. **re-runs the first queries of the recorded benchmark** and checks that every number matches what is on disk, for the released explanation and for the direction-corrected one,
4. **shows the message-direction bug** in the released explanation code on three queries,
5. re-runs the neuromodulation experiment and compares it the same way,
6. reproduces link prediction on the full CoDEx-S test set,
7. rebuilds the benchmark's reliability analysis and figures from the shipped results,
8. runs the full neuromodulation experiment, saving to Google Drive as it goes.

**Runtime:** a CPU runtime is enough, and the default one is fine. Nothing here trains, and the code runs on CPU. On a GPU runtime the GPU is hidden, so ULTRA's kernel compiles for CPU only.

**Time:** about 3 minutes of setup, then 1 to 2 minutes per demo run.

## 1. Get the code

Pick where the project code comes from. With `'drive'` it mounts your Google Drive (Colab asks you to allow this) and uses the newest `CAP6938_pathfaith_code.zip` in the `CAP6938 pathfaith` folder. With `'upload'` it asks you to upload the zip.

In [ ]:
# 'upload': upload CAP6938_pathfaith_code.zip when prompted
# 'drive' : read the zip from Google Drive, at DRIVE_ZIP or the newest copy next to it
# 'git'   : clone a repository, at GIT_URL
SOURCE = 'upload'
DRIVE_ZIP = '/content/drive/MyDrive/CAP6938 pathfaith/CAP6938_pathfaith_code.zip'
GIT_URL = ''

import os, subprocess, zipfile
WORK = os.environ.get('PATHFAITH_WORKDIR', '/content/pathfaith')
LOCAL_ZIP = os.environ.get('PATHFAITH_ZIP')   # only used when testing outside Colab

STAMP = os.path.join(WORK, '.zip_used')
zip_path = None
if True:
    os.makedirs(WORK, exist_ok=True)
    if LOCAL_ZIP:
        zip_path = LOCAL_ZIP
    elif SOURCE == 'upload':
        from google.colab import files
        up = files.upload()
        zip_path = os.path.join('/content', next(iter(up)))
    elif SOURCE == 'drive':
        import glob
        from google.colab import drive
        if not os.path.ismount('/content/drive'):
            drive.mount('/content/drive')
        found = set(glob.glob('/content/drive/MyDrive/CAP6938 pathfaith/CAP6938_pathfaith_code*.zip')
                    + glob.glob('/content/drive/MyDrive/CAP6938_pathfaith_code*.zip'))
        if os.path.exists(DRIVE_ZIP):
            found.add(DRIVE_ZIP)
        if not found:
            raise FileNotFoundError('no CAP6938_pathfaith_code.zip in My Drive; set DRIVE_ZIP or use SOURCE = "upload"')
        zip_path = max(found, key=os.path.getmtime)   # a re-uploaded zip wins over an older one
        print('using', zip_path)
    elif SOURCE == 'git':
        subprocess.run(['git', 'clone', '-q', GIT_URL, WORK], check=True)
        zip_path = None
    else:
        raise ValueError(SOURCE)
    if zip_path:
        tag = f'{zip_path} {os.path.getmtime(zip_path)}'
        if not os.path.exists(STAMP) or open(STAMP).read() != tag:
            with zipfile.ZipFile(zip_path) as z:     # a newer zip replaces the old code
                z.extractall(WORK)
            open(STAMP, 'w').write(tag)
            print('extracted', os.path.basename(zip_path))
        else:
            print('code already up to date')
    print('code in', WORK, ':', sorted(x for x in os.listdir(WORK) if not x.startswith('.')))
os.chdir(WORK)
DRIVE_DIR = os.path.dirname(zip_path) if zip_path and zip_path.startswith('/content/drive') else None

## 2. Install

PyTorch, NumPy and matplotlib are already on Colab. This adds PyTorch Geometric, clones ULTRA at the commit the project used (its repository ships the checkpoint), and installs `torch_scatter`.

`torch_scatter` is a compiled extension. If PyG publishes a wheel for Colab's PyTorch, that wheel is used. If not, the cell installs `colab/torch_scatter_shim` instead, a small pure-PyTorch package with the same functions. It agrees exactly with the compiled extension on every function ULTRA calls, and section 5 checks it against the recorded benchmark anyway.

In [ ]:
import os, sys, subprocess
os.environ['CUDA_VISIBLE_DEVICES'] = ''        # the project runs on CPU; keep ULTRA's kernel build CPU-only
os.environ['PYTHONWARNINGS'] = 'ignore'        # library deprecation chatter, in every command below
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--disable-pip-version-check', 'torch_geometric', 'easydict', 'ninja'], check=True)

import torch
major, minor = torch.__version__.split('+')[0].split('.')[:2]
whl_index = f'https://data.pyg.org/whl/torch-{major}.{minor}.0+cpu.html'

def scatter_works():
    return subprocess.run([sys.executable, '-c', 'import torch_scatter; torch_scatter.scatter_add'],
                          capture_output=True).returncode == 0

if not scatter_works() and os.environ.get('PATHFAITH_FORCE_SHIM') != '1':
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--disable-pip-version-check', '--only-binary=:all:',
                    'torch_scatter', '-f', whl_index], capture_output=True)
if not scatter_works() or os.environ.get('PATHFAITH_FORCE_SHIM') == '1':
    subprocess.run([sys.executable, '-m', 'pip', 'uninstall', '-y', '-q', '--disable-pip-version-check', 'torch_scatter'], capture_output=True)
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--disable-pip-version-check', './colab/torch_scatter_shim'], check=True)
    print('torch_scatter: no prebuilt wheel for torch', torch.__version__, '-> using the pure-PyTorch shim')
else:
    import importlib.metadata as md
    print('torch_scatter: prebuilt wheel', md.version('torch_scatter'))

if not os.path.exists('ultra/ckpts/ultra_4g.pth'):
    subprocess.run(['git', 'clone', '-q', 'https://github.com/DeepGraphLearning/ULTRA.git', 'ultra'], check=True)
    subprocess.run(['git', '-C', 'ultra', 'checkout', '-q', '427966ad8ed60420eef034063d44f3153addff90'], check=True)
print('python', sys.version.split()[0], '| torch', torch.__version__, '| ULTRA', 
      subprocess.run(['git', '-C', 'ultra', 'rev-parse', '--short', 'HEAD'], capture_output=True, text=True).stdout.strip())

## 3. Load the frozen checkpoint once

This downloads CoDEx-S, loads `ultra_4g` and runs one forward pass, which compiles ULTRA's `rspmm` CPU kernel. The compile takes about a minute and only happens once per session, so the demo below starts straight away.

In [ ]:
!python -m pathfaith.warmup --dataset CoDExSmall

## 4. Live demo

For each query the demo prints the answer and NBFNet's own top paths, then intervenes on the graph. Each line below the explanation is a fresh forward pass on an edited graph:

* **delete those paths**: the answer should drop if the explanation is load-bearing
* **delete as many random edges**: the control, which should leave the answer where it was
* **delete only the busiest edge**: the one edge most of the paths share, and how much of the effect it carries alone
* **reroute the last hop**: does probability follow the edit to the new entity?
* **neuromodulate**: the main relation channel the explanation uses, turned down to half and to zero, and whether the gradient at full gain had the right sign

Under the explanation, one line shows the bug: the released gradient is taken on a model that gives the answer a different probability from the deployed model.

It ends with how many queries finished every intervention, then reads the full benchmark off disk.

In [ ]:
!python -m pathfaith.demo --n 5 --seed 0 --color always

A second run with a different seed draws different queries. Run it as often as you like with any seed.

In [ ]:
!python -m pathfaith.demo --n 3 --seed 7 --no-benchmark --color always

Optional: the same demo on WN18RR, with WordNet synset names. The first run downloads WN18RR, which takes about a minute.

In [ ]:
!python -m pathfaith.demo --dataset WN18RR --n 2 --no-benchmark --color always

## 5. Re-run the recorded benchmark and compare

The benchmark's queries come in an order fixed by the seed, and the model is frozen and deterministic. So re-running the first queries here should give exactly the numbers recorded in `results/`, field by field: the starting probability and rank, all 20 single-path deletions, the joint deletion, the random control, keep-only, the busiest edge and the rewire. Ten queries take three to five minutes.

In [ ]:
N = 10
!rm -f colab_check/paths_codexsmall.jsonl
!python -m pathfaith.run_experiment --dataset CoDExSmall --n {N} --pool 20 --topk 5 --seed 0 --path-controls 0 --out colab_check/paths_codexsmall.jsonl
!python -m pathfaith.compare_runs --new colab_check/paths_codexsmall.jsonl --ref results/paths_codexsmall_seed0.jsonl

The same check for the direction-corrected explanation, on five queries.

In [ ]:
!rm -f colab_check/paths_codexsmall_fixed.jsonl
!python -m pathfaith.run_experiment --dataset CoDExSmall --n 5 --pool 20 --topk 5 --seed 0 --path-controls 0 --corrected --out colab_check/paths_codexsmall_fixed.jsonl
!python -m pathfaith.compare_runs --new colab_check/paths_codexsmall_fixed.jsonl --ref results/paths_codexsmall_fixed_seed0.jsonl

## 5b. The message-direction bug

ULTRA's fused `rspmm` kernel, which the model uses to make predictions, sends each message from `edge_index[1]` to `edge_index[0]`. When edge weights need a gradient, which is when NBFNet's `visualize()` runs, the layers fall back to PyTorch Geometric's generic message passing, which sends it the other way. So the released explanation differentiates a different model. For each query this prints the largest score gap between the two code paths (released, and on the reversed graph), and, along a random direction, the deployed model's measured change in p against each gradient. About a minute.

In [ ]:
!python -m pathfaith.direction_check --dataset CoDExSmall --n 3 --keep-scores 0 --out colab_check/direction_codexsmall.jsonl

## 6. Relation channels as neuromodulators

Following Bull et al. (NeurIPS 2025), each relation type is treated as a neuromodulatory channel: ULTRA's relation pass gives one vector per relation, and it multiplies every message along edges of that type. Each channel gets a gain, turned down from 1 to 0 like a dose, and the dose response is compared with the gradient at full gain. This runs the first 3 CoDEx-S queries and, when the zip carries recorded results, compares every number with them. About two minutes. Section 9 runs the full experiment.

In [ ]:
import os
!rm -f colab_check/neuromod_codexsmall.jsonl
!python -m pathfaith.neuromod --dataset CoDExSmall --n 3 --out colab_check/neuromod_codexsmall.jsonl
if os.path.exists('results/neuromod_codexsmall_seed0.jsonl'):
    !python -m pathfaith.compare_runs --new colab_check/neuromod_codexsmall.jsonl --ref results/neuromod_codexsmall_seed0.jsonl
else:
    print('this zip has no recorded neuromodulation results yet; section 9 produces them')
    !python -m pathfaith.analyze_neuromod colab_check/neuromod_codexsmall.jsonl

## 7. Reproduce link prediction

Filtered MRR and Hits@k on the **full** CoDEx-S test set, both directions scored, with ULTRA's own evaluation code. The ULTRA paper reports MRR 0.472 for its pre-trained model on CoDEx-S; the 4g checkpoint adds NELL-995 to that model's pre-training mix. About two minutes.

In [ ]:
!python -m pathfaith.run_baseline --dataset CoDExSmall --n 0 --batch-size 8 --out colab_check/baseline_codexsmall.json 2>&1 | grep -E '"(n_triples|mrr|hits@1|hits@10)"'

## 8. The benchmark's reliability, from the shipped results

No model calls here: this reads the per-query records in `results/`. Because the query order was fixed in advance, the first *n* queries are an honest sample of size *n*, so each curve shows a headline estimate settling as queries are added. The dots at *n* = 5 are every disjoint block of five queries, which is what a five-query benchmark could have reported.

In [ ]:
!python -m pathfaith.reliability
from pathfaith.figures import fig_convergence
from IPython.display import Image, display
fig_convergence('colab_check/convergence.png', figsize=(11.07, 6.32), big=True)
display(Image('colab_check/convergence.png'))

## 9. Run the full neuromodulation experiment

The same experiment on 200 queries per graph, in the benchmark's fixed order. Results are written to Google Drive (`CAP6938 pathfaith/colab_results/`) one query at a time, so if Colab disconnects, run this cell again and it carries on where it stopped. CoDEx-S and NELL23k take about 1 to 1.5 hours each on a Colab CPU. WN18RR is slower, and FB15k-237 much slower (about 3 minutes per query), so they are left out by default.

In [ ]:
import os
GRAPHS = ['CoDExSmall', 'NELL23k']        # add 'WN18RR', 'FB15k237' if there is time
N = 200
OUT = os.path.join(DRIVE_DIR, 'colab_results') if DRIVE_DIR else 'colab_check'
os.makedirs(OUT, exist_ok=True)
print('writing to', OUT)
for ds in GRAPHS:
    out = os.path.join(OUT, f'neuromod_{ds.lower()}_seed0.jsonl')
    !python -m pathfaith.neuromod --dataset {ds} --n {N} --out "{out}"

Summary and dose-response curves from what has finished, and, where the zip carries the recorded run, a query-by-query comparison with it.

In [ ]:
import glob, os, shutil
for f in sorted(glob.glob(os.path.join(OUT, 'neuromod_*_seed0.jsonl'))):
    ref = os.path.join('results', os.path.basename(f))
    !python -m pathfaith.analyze_neuromod "{f}"
    if os.path.exists(ref):
        !python -m pathfaith.compare_runs --new "{f}" --ref "{ref}"
os.makedirs('colab_check/nm', exist_ok=True)
for f in glob.glob(os.path.join(OUT, 'neuromod_*_seed0.jsonl')):
    shutil.copy(f, 'colab_check/nm/')
from pathfaith.figures import fig_dose_response
from IPython.display import Image, display
fig_dose_response('colab_check/dose_response.png', results_dir='colab_check/nm', figsize=(11.07, 6.0), big=True)
display(Image('colab_check/dose_response.png'))

## Going further

* `./run_stage.sh STAGE DATASET N` runs one stage of the benchmark: `sweep` (released explanation), `fixed` (direction-corrected), `fixed3g` (corrected, on ULTRA's 3-graph checkpoint), `ig` (integrated gradients), `neuromod`, `direction` or `baseline`. The recorded benchmark is 200 queries on each of four graphs. Every stage resumes where it stopped, so it can be run in pieces across Colab sessions with `results/` kept on Google Drive.
* `./finalize.sh` rebuilds the numbers, figures, report, slides, narration and video from `results/`. The report and slides need LaTeX (`apt-get install texlive-latex-extra texlive-fonts-recommended`).